# DFU Repair-7 v1.5 — Preserve 38 Good Trials
Pinned wrapper around the exact v1.4 notebook. Canonicalizes Base64 padding only; existing SHA checks still protect the embedded evidence.


In [ ]:
import base64, hashlib, json, urllib.request

V14_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/c3ed0a49e94acd651981ad8e6a5809c9c61aafaf/notebooks/DFU_Repair7_Only_Preserve38_v1_4_Colab.ipynb"
EXPECTED_V14_SHA256 = "3ab85228c038bb0592561157ba961e3ad303b403c53db89d6e0cd0a99cafc11a"

# Canonicalize Base64 padding before decoding. This fixes only transport padding;
# the existing SHA-256 checks inside v1.4 still reject any actual data corruption.
_ORIGINAL_B64DECODE = base64.b64decode

def _canonical_b64decode(s, altchars=None, validate=False):
    if isinstance(s, str):
        raw = s.encode("ascii")
    else:
        raw = bytes(s)
    raw = b"".join(raw.split())
    core = raw.rstrip(b"=")
    raw = core + (b"=" * ((4 - len(core) % 4) % 4))
    return _ORIGINAL_B64DECODE(raw, altchars=altchars, validate=validate)

base64.b64decode = _canonical_b64decode

v14_bytes = urllib.request.urlopen(V14_URL, timeout=120).read()
actual_sha = hashlib.sha256(v14_bytes).hexdigest()
if actual_sha != EXPECTED_V14_SHA256:
    raise RuntimeError(f"Pinned v1.4 notebook SHA mismatch: {actual_sha} != {EXPECTED_V14_SHA256}")

v14_nb = json.loads(v14_bytes.decode("utf-8"))
code_cells = [c for c in v14_nb.get("cells", []) if c.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected exactly one v1.4 executable cell, found {len(code_cells)}")
v14_code = "".join(code_cells[0]["source"])
if "Pinned Repair-7 v1.4 loader verification: PASS" not in v14_code:
    raise RuntimeError("Unexpected v1.4 loader content; refusing execution.")

print("Pinned Repair-7 v1.5 wrapper verification: PASS")
print("Canonical Base64 padding guard: INSTALLED")
exec(compile(v14_code, "DFU_Repair7_v1_5_wrapper.py", "exec"), globals())
